In [ ]:
import os
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from typing import Dict, Tuple, List

# Root folder path
ROOT_PATH = Path(r"C:\Users\grupp\Downloads\belgium")

# Selection strategy for representative file per type: "first", "largest", or "newest"
REP_SELECTION = "first"

# Map synonymous extensions to a normalized label
EXT_NORMALIZATION = {
    ".tiff": ".tif",
    ".jpeg": ".jpg",
    ".htm": ".html",
    ".yml": ".yaml",
}

def normalize_ext(ext: str) -> str:
    """Return a normalized extension label. Empty becomes '[no extension]'."""
    if not ext:
        return "[no extension]"
    ext = ext.lower()
    return EXT_NORMALIZATION.get(ext, ext)

def print_folder_structure(folder: Path, indent: int = 0) -> None:
    """Recursively prints only the folder structure (no file listings)."""
    print("    " * indent + f"[{folder.name}]")
    try:
        for item in sorted(folder.iterdir()):
            if item.is_dir():
                print_folder_structure(item, indent + 1)
    except PermissionError:
        print("    " * (indent + 1) + "[Permission Denied]")

def walk_files(folder: Path) -> List[Path]:
    """Yield all files under folder recursively."""
    for dirpath, _, filenames in os.walk(folder):
        d = Path(dirpath)
        for fn in filenames:
            yield d / fn

def collect_type_stats(folder: Path) -> Tuple[Dict[str, Dict], Dict[str, Path]]:
    """
    Build per-type stats:
      - count
      - total_size
      - representative file selected by REP_SELECTION
    Returns (stats_by_type, representative_by_type).
    """
    stats = defaultdict(lambda: {"count": 0, "total_size": 0})
    representative: Dict[str, Path] = {}

    for fp in walk_files(folder):
        try:
            st = fp.stat()
        except (OSError, PermissionError):
            continue

        ext = normalize_ext(fp.suffix)
        stats[ext]["count"] += 1
        stats[ext]["total_size"] += st.st_size

        # Maintain one representative by selection strategy
        if ext not in representative:
            representative[ext] = fp
        else:
            current = representative[ext]
            try:
                cur_stat = current.stat()
            except (OSError, PermissionError):
                representative[ext] = fp
                continue

            if REP_SELECTION == "largest":
                if st.st_size > cur_stat.st_size:
                    representative[ext] = fp
            elif REP_SELECTION == "newest":
                if st.st_mtime > cur_stat.st_mtime:
                    representative[ext] = fp
            # default "first": keep existing
    return stats, representative

def fmt_bytes(n: int) -> str:
    """Human-readable byte sizes."""
    units = ["B", "KB", "MB", "GB", "TB"]
    size = float(n)
    u = 0
    while size >= 1024 and u < len(units) - 1:
        size /= 1024.0
        u += 1
    return f"{size:.2f} {units[u]}"

def print_file_info(file_path: Path) -> None:
    """Prints detailed information about a file."""
    try:
        stat = file_path.stat()
    except (OSError, PermissionError):
        print(f"File: {file_path}")
        print("  [Unable to access file metadata]")
        print("-" * 50)
        return

    size_str = fmt_bytes(stat.st_size)
    mod_time = datetime.fromtimestamp(stat.st_mtime).strftime("%Y-%m-%d %H:%M:%S")
    print(f"File: {file_path}")
    print(f"  Extension: {normalize_ext(file_path.suffix)}")
    print(f"  Size: {size_str}")
    print(f"  Last Modified: {mod_time}")
    print("-" * 50)

if __name__ == "__main__":
    print("=== Folder Structure ===")
    print_folder_structure(ROOT_PATH)

    stats_by_type, rep_by_type = collect_type_stats(ROOT_PATH)

    # Sort extensions alphabetically, but put '[no extension]' last
    def sort_key(ext: str) -> Tuple[int, str]:
        return (1, "") if ext == "[no extension]" else (0, ext)

    print("\n=== Unique File Types (Counts and Totals) ===")
    for ext in sorted(stats_by_type.keys(), key=sort_key):
        info = stats_by_type[ext]
        print(f"{ext}: {info['count']} files, total {fmt_bytes(info['total_size'])}")

    print("\n=== One Example File Per Type ===")
    for ext in sorted(rep_by_type.keys(), key=sort_key):
        print_file_info(rep_by_type[ext])


# Create Bounding Boxes for Tifs in a Folder

In [8]:
import os
from pathlib import Path
import rasterio
from rasterio.warp import transform_bounds
import geopandas as gpd
from shapely.geometry import box

def generate_bounding_boxes(folder_path):
    folder_path = Path(folder_path)
    tif_files = list(folder_path.glob("*.tif"))

    if not tif_files:
        print(f"No TIFF files found in {folder_path}")
        return

    bbox_records = []
    for tif in tif_files:
        try:
            with rasterio.open(tif) as src:
                bounds = src.bounds
                crs = src.crs
                # Transform bounds to EPSG:4326 for shapefile output
                minx, miny, maxx, maxy = transform_bounds(crs, "EPSG:4326",
                                                          bounds.left, bounds.bottom,
                                                          bounds.right, bounds.top)
                geometry = box(minx, miny, maxx, maxy)
                bbox_records.append({"filename": tif.name, "geometry": geometry})
        except Exception as e:
            print(f"Error reading {tif}: {e}")

    if bbox_records:
        gdf = gpd.GeoDataFrame(bbox_records, crs="EPSG:4326")
        output_name = folder_path.name + "BoundingBox.shp"
        output_path = folder_path / output_name
        gdf.to_file(output_path)
        print(f"Bounding boxes saved to {output_path}")
    else:
        print("No bounding boxes generated.")

if __name__ == "__main__":
    # Change this to your folder of TIFFs
    tif_folder = r"C:\Users\grupp\Desktop\Mato Grosso AL MVP for tech note\MG AL true sets\chiquitanorandom"
    generate_bounding_boxes(tif_folder)


Bounding boxes saved to C:\Users\grupp\Desktop\Mato Grosso AL MVP for tech note\MG AL true sets\chiquitanorandom\chiquitanorandomBoundingBox.shp


# Create Masks

In [ ]:
# Split windows + create 2-class, 3-class, and instance masks


import os
from pathlib import Path
from typing import Dict, Tuple, List, Optional
import numpy as np
import geopandas as gpd
from shapely.geometry import box as shapely_box
import rasterio
from rasterio.features import rasterize
from datetime import datetime
from collections import defaultdict
from tqdm import tqdm

# ------------------------------------------------------------------------------------
# CONFIGURE PATHS (updated for each folder you wanna do)
# ------------------------------------------------------------------------------------
ROOT_DIR = Path(r"C:\Users\grupp\Desktop\Mato Grosso AL MVP for tech note\MG AL true sets\SouthAmericaSoy")
BBOX_SHP = ROOT_DIR / "SouthAmericaSoy_Random_BoundingBoxes.shp"
FIELDS_SHP = ROOT_DIR / "SouthAmericaSoy_Random_FieldsFinalQA.shp"

# Output structure
DIR_WINDOW_A = ROOT_DIR / "s2_images" / "window_a"
DIR_WINDOW_B = ROOT_DIR / "s2_images" / "window_b"
DIR_MASKS_2C = ROOT_DIR / "label_masks" / "semantic_2class"
DIR_MASKS_3C = ROOT_DIR / "label_masks" / "semantic_3class"
DIR_MASKS_INS = ROOT_DIR / "label_masks" / "instance"

for d in [DIR_WINDOW_A, DIR_WINDOW_B, DIR_MASKS_2C, DIR_MASKS_3C, DIR_MASKS_INS]:
    d.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------------
# UTILITIES
# ------------------------------------------------------------------------------------
def fmt_bytes(n: int) -> str:
    units = ["B", "KB", "MB", "GB", "TB"]
    size = float(n)
    u = 0
    while size >= 1024 and u < len(units) - 1:
        size /= 1024.0
        u += 1
    return f"{size:.2f} {units[u]}"

def normalize_ext(ext: str) -> str:
    if not ext:
        return "[no extension]"
    ext = ext.lower()
    mapping = {".tiff": ".tif", ".jpeg": ".jpg", ".htm": ".html", ".yml": ".yaml"}
    return mapping.get(ext, ext)

def print_folder_structure(folder: Path, indent: int = 0) -> None:
    print("    " * indent + f"[{folder.name}]")
    try:
        for item in sorted(folder.iterdir()):
            if item.is_dir():
                print_folder_structure(item, indent + 1)
    except PermissionError:
        print("    " * (indent + 1) + "[Permission Denied]")

def walk_files(folder: Path) -> List[Path]:
    for dirpath, _, filenames in os.walk(folder):
        d = Path(dirpath)
        for fn in filenames:
            yield d / fn

def collect_type_stats(folder: Path):
    stats = defaultdict(lambda: {"count": 0, "total_size": 0})
    representative: Dict[str, Path] = {}
    for fp in walk_files(folder):
        try:
            st = fp.stat()
        except (OSError, PermissionError):
            continue
        ext = normalize_ext(fp.suffix)
        stats[ext]["count"] += 1
        stats[ext]["total_size"] += st.st_size
        if ext not in representative:
            representative[ext] = fp
    return stats, representative

def print_file_info(file_path: Path) -> None:
    try:
        stat = file_path.stat()
        size_str = fmt_bytes(stat.st_size)
        mod_time = datetime.fromtimestamp(stat.st_mtime).strftime("%Y-%m-%d %H:%M:%S")
        print(f"File: {file_path}")
        print(f"  Extension: {normalize_ext(file_path.suffix)}")
        print(f"  Size: {size_str}")
        print(f"  Last Modified: {mod_time}")
        print("-" * 50)
    except (OSError, PermissionError):
        print(f"File: {file_path}")
        print("  [Unable to access file metadata]")
        print("-" * 50)

def get_raster_bounds_polygon(src: rasterio.io.DatasetReader):
    left, bottom, right, top = src.bounds
    return shapely_box(left, bottom, right, top)

def match_bbox_for_raster(bbox_gdf: gpd.GeoDataFrame, raster_poly, raster_crs):
    if bbox_gdf.crs != raster_crs:
        bbox_gdf = bbox_gdf.to_crs(raster_crs)
    candidates = bbox_gdf[bbox_gdf.intersects(raster_poly)]
    if candidates.empty:
        return raster_poly
    candidates = candidates.copy()
    candidates["__intersect_area"] = candidates.geometry.intersection(raster_poly).area
    best = candidates.sort_values("__intersect_area", ascending=False).iloc[0]
    return best.geometry

def polygons_for_raster(fields_gdf: gpd.GeoDataFrame, bbox_geom, raster_crs) -> gpd.GeoDataFrame:
    gdf = fields_gdf
    if gdf.crs != raster_crs:
        gdf = gdf.to_crs(raster_crs)
    inter = gdf[gdf.intersects(bbox_geom)]
    if inter.empty:
        return inter
    try:
        inter = gpd.clip(inter, gpd.GeoDataFrame(geometry=[bbox_geom], crs=raster_crs))
    except Exception:
        pass
    return inter

def write_window_tif(out_path: Path, src: rasterio.io.DatasetReader, band_indices: List[int]):
    meta = src.meta.copy()
    meta.update({
        "count": len(band_indices),
        "dtype": src.dtypes[0],
        "compress": "deflate",
        "tiled": True,
        "blockxsize": 256,
        "blockysize": 256,
        "nodata": src.nodata
    })
    data = src.read(band_indices)
    with rasterio.open(out_path, "w", **meta) as dst:
        dst.write(data)

def write_semantic_2class(out_path: Path, shape_hw, transform, crs, polygons_gdf):
    height, width = shape_hw
    mask = np.zeros((height, width), dtype=np.uint8)
    if not polygons_gdf.empty:
        shapes = [(geom, 1) for geom in polygons_gdf.geometry if geom and not geom.is_empty]
        if shapes:
            mask = rasterize(shapes, out_shape=(height, width), transform=transform, fill=0, dtype="uint8")
    meta = {"driver": "GTiff", "height": height, "width": width, "count": 1, "dtype": "uint8",
            "crs": crs, "transform": transform, "compress": "deflate", "tiled": True,
            "blockxsize": 256, "blockysize": 256, "nodata": 0}
    with rasterio.open(out_path, "w", **meta) as dst:
        dst.write(mask, 1)

def write_semantic_3class(out_path: Path, shape_hw, transform, crs, polygons_gdf, boundary_px=1):
    height, width = shape_hw
    mask = np.zeros((height, width), dtype=np.uint8)
    if not polygons_gdf.empty:
        # Interiors
        shapes_int = [(geom, 1) for geom in polygons_gdf.geometry if geom and not geom.is_empty]
        if shapes_int:
            interior = rasterize(shapes_int, out_shape=(height, width), transform=transform, fill=0, dtype="uint8")
            mask[interior == 1] = 1
        # Boundaries
        px_size = max(abs(transform.a), abs(transform.e))
        b_geom = [geom.boundary.buffer(px_size * boundary_px, cap_style=1, join_style=2)
                  for geom in polygons_gdf.geometry if geom and not geom.is_empty]
        shapes_bound = [(g, 2) for g in b_geom if not g.is_empty]
        if shapes_bound:
            boundary = rasterize(shapes_bound, out_shape=(height, width), transform=transform, fill=0, dtype="uint8")
            mask[(boundary == 2) & (mask == 0)] = 2
    meta = {"driver": "GTiff", "height": height, "width": width, "count": 1, "dtype": "uint8",
            "crs": crs, "transform": transform, "compress": "deflate", "tiled": True,
            "blockxsize": 256, "blockysize": 256, "nodata": 0}
    with rasterio.open(out_path, "w", **meta) as dst:
        dst.write(mask, 1)

def write_instance(out_path: Path, shape_hw, transform, crs, polygons_gdf, id_field=None):
    height, width = shape_hw
    mask = np.zeros((height, width), dtype=np.uint32)
    if not polygons_gdf.empty:
        if id_field and id_field in polygons_gdf.columns:
            polygons_gdf = polygons_gdf.copy()
            polygons_gdf["__id"] = [int(v) if v is not None else 0 for v in polygons_gdf[id_field]]
        else:
            polygons_gdf = polygons_gdf.copy()
            polygons_gdf["__id"] = range(1, len(polygons_gdf) + 1)
        shapes = [(g, i) for g, i in zip(polygons_gdf.geometry, polygons_gdf["__id"]) if g and not g.is_empty]
        if shapes:
            mask = rasterize(shapes, out_shape=(height, width), transform=transform, fill=0, dtype="uint32")
    meta = {"driver": "GTiff", "height": height, "width": width, "count": 1, "dtype": "uint32",
            "crs": crs, "transform": transform, "compress": "deflate", "tiled": True,
            "blockxsize": 256, "blockysize": 256, "nodata": 0}
    with rasterio.open(out_path, "w", **meta) as dst:
        dst.write(mask, 1)

# ------------------------------------------------------------------------------------
# LOAD VECTORS
# ------------------------------------------------------------------------------------
bbox_gdf = gpd.read_file(BBOX_SHP)
fields_gdf = gpd.read_file(FIELDS_SHP)

# ------------------------------------------------------------------------------------
# PROCESS RASTERS
# ------------------------------------------------------------------------------------
tif_list = sorted([p for p in ROOT_DIR.glob("*.tif") if p.is_file()])

for tif_path in tqdm(tif_list, desc="Processing rasters", unit="tif"):
    with rasterio.open(tif_path) as src:
        if src.count < 8:
            raise ValueError(f"{tif_path.name} has {src.count} bands; expected 8.")
        raster_poly = get_raster_bounds_polygon(src)
        raster_crs = src.crs
        bbox_geom = match_bbox_for_raster(bbox_gdf, raster_poly, raster_crs)
        inter_poly_gdf = polygons_for_raster(fields_gdf, bbox_geom, raster_crs)
        # Windows
        write_window_tif(DIR_WINDOW_A / tif_path.name, src, [1, 2, 3, 4])
        write_window_tif(DIR_WINDOW_B / tif_path.name, src, [5, 6, 7, 8])
        # Masks
        shape_hw = (src.height, src.width)
        transform = src.transform
        crs = src.crs
        write_semantic_2class(DIR_MASKS_2C / tif_path.name, shape_hw, transform, crs, inter_poly_gdf)
        write_semantic_3class(DIR_MASKS_3C / tif_path.name, shape_hw, transform, crs, inter_poly_gdf)
        id_field = "id" if "id" in inter_poly_gdf.columns else None
        write_instance(DIR_MASKS_INS / tif_path.name, shape_hw, transform, crs, inter_poly_gdf, id_field)

# ------------------------------------------------------------------------------------
# REPORT
# ------------------------------------------------------------------------------------
print("=== Folder Structure ===")
print_folder_structure(ROOT_DIR)

stats_by_type, rep_by_type = collect_type_stats(ROOT_DIR)
def sort_key(ext: str):
    return (1, "") if ext == "[no extension]" else (0, ext)

print("\n=== Unique File Types (Counts and Totals) ===")
for ext in sorted(stats_by_type.keys(), key=sort_key):
    info = stats_by_type[ext]
    print(f"{ext}: {info['count']} files, total {fmt_bytes(info['total_size'])}")

print("\n=== One Example File Per Type ===")
for ext in sorted(rep_by_type.keys(), key=sort_key):
    print_file_info(rep_by_type[ext])


Processing rasters:   0%|          | 0/30 [00:00<?, ?tif/s]C:\Users\grupp\AppData\Local\Temp\ipykernel_27276\3441839462.py:111: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  candidates["__intersect_area"] = candidates.geometry.intersection(raster_poly).area
C:\Users\grupp\AppData\Local\Temp\ipykernel_27276\3441839462.py:111: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  candidates["__intersect_area"] = candidates.geometry.intersection(raster_poly).area
Processing rasters:   7%|▋         | 2/30 [00:00<00:02, 10.29tif/s]C:\Users\grupp\AppData\Local\Temp\ipykernel_27276\3441839462.py:111: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geomet

=== Folder Structure ===
[SouthAmericaSoy]
    [label_masks]
        [instance]
        [semantic_2class]
        [semantic_3class]
    [s2_images]
        [window_a]
        [window_b]

=== Unique File Types (Counts and Totals) ===
.cpg: 2 files, total 10.00 B
.dbf: 2 files, total 81.75 KB
.prj: 2 files, total 570.00 B
.sbn: 2 files, total 13.69 KB
.sbx: 2 files, total 520.00 B
.shp: 2 files, total 1.09 MB
.shx: 2 files, total 11.35 KB
.tif: 180 files, total 56.61 MB
.xml: 2 files, total 13.11 KB

=== One Example File Per Type ===
File: C:\Users\grupp\Desktop\Mato Grosso AL MVP for tech note\MG AL true sets\SouthAmericaSoy\SouthAmericaSoy_Random_BoundingBoxes.cpg
  Extension: .cpg
  Size: 5.00 B
  Last Modified: 2025-08-13 14:46:16
--------------------------------------------------
File: C:\Users\grupp\Desktop\Mato Grosso AL MVP for tech note\MG AL true sets\SouthAmericaSoy\SouthAmericaSoy_Random_BoundingBoxes.dbf
  Extension: .dbf
  Size: 2.44 KB
  Last Modified: 2025-08-13 14:46:16


# Create Split-Level Folders

In [ ]:
# JUPYTER NOTEBOOK SINGLE-CELL
# Split a dataset into Training / Testing / Validation by sampling TIFF basenames
# from label_masks/instance, then copy the corresponding files from:
#   - label_masks/instance
#   - label_masks/semantic_2class
#   - label_masks/semantic_3_class
#   - s2_images/window_a
#   - s2_images/window_b
# to new sibling folders that preserve the same internal folder structure.
#
# Process:
# 1) Creates <Base>_Training, <Base>_Testing, <Base>_Validation.
# 2) Gathers all .tif/.tiff filenames in <Base>\label_masks\instance.
# 3) Randomly samples X for Training, X for Testing, X for Validation.
# 4) Copies the selected files for ALL modalities (instance, semantic_2class, semantic_3_class,
#    window_a, window_b) into the corresponding split folder, preserving the subfolder structure.
#
# Requirements: Python 3.8+, tqdm


import os
import random
import shutil
from pathlib import Path
from typing import List, Dict, Tuple
from tqdm import tqdm
from datetime import datetime

# ----------------------------
# USER CONFIGURATION
# ----------------------------
BASE_FOLDERS = [
    r"C:\Users\grupp\Desktop\Mato Grosso AL MVP for tech note\MG AL true sets\MatoGrosso_Brazil_Confidences",
    r"C:\Users\grupp\Desktop\Mato Grosso AL MVP for tech note\MG AL true sets\MatoGrosso_Brazil_LCDI",
    r"C:\Users\grupp\Desktop\Mato Grosso AL MVP for tech note\MG AL true sets\MatoGrosso_Brazil_LCDIB",
    r"C:\Users\grupp\Desktop\Mato Grosso AL MVP for tech note\MG AL true sets\MatoGrosso_Brazil_Random",
]

# Desired split counts (must sum to the number you want to select per dataset)
TRAIN_COUNT = 60
TEST_COUNT  = 20
VAL_COUNT   = 20

# Random seed for reproducibility
RANDOM_SEED = 42

# If True, will overwrite existing files in the destination; if False, will skip existing files
OVERWRITE = False

# ----------------------------
# CONSTANTS / RELATIVE PATHS
# ----------------------------
RELATIVE_FOLDERS = {
    "instance":        Path("label_masks") / "instance",
    "semantic_2class": Path("label_masks") / "semantic_2class",
    "semantic_3_class":Path("label_masks") / "semantic_3_class",
    "window_a":        Path("s2_images")   / "window_a",
    "window_b":        Path("s2_images")   / "window_b",
}
VALID_EXTS = (".tif", ".tiff")  # We will look for these when copying

# ----------------------------
# HELPERS
# ----------------------------
def ensure_split_dirs(base_dir: Path) -> Dict[str, Dict[str, Path]]:
    """
    For a given base folder, create sibling Training/Testing/Validation folders.
    Return a mapping of split -> modality -> destination path.
    """
    parent = base_dir.parent
    name = base_dir.name

    split_dirs = {
        "Training":   parent / f"{name}_Training",
        "Testing":    parent / f"{name}_Testing",
        "Validation": parent / f"{name}_Validation",
    }
    for split, p in split_dirs.items():
        p.mkdir(parents=True, exist_ok=True)
        # Create subfolders
        for rel in RELATIVE_FOLDERS.values():
            (p / rel).mkdir(parents=True, exist_ok=True)

    # Return structured map for convenience
    out = {}
    for split, root in split_dirs.items():
        out[split] = {k: root / v for k, v in RELATIVE_FOLDERS.items()}
    return out

def list_instance_basenames(base_dir: Path) -> List[str]:
    """
    List unique basenames (without extension) of all .tif/.tiff in label_masks/instance.
    """
    inst_dir = base_dir / RELATIVE_FOLDERS["instance"]
    if not inst_dir.exists():
        raise FileNotFoundError(f"Instance folder not found: {inst_dir}")

    basenames = set()
    for ext in VALID_EXTS:
        for p in inst_dir.glob(f"*{ext}"):
            basenames.add(p.stem)
    return sorted(basenames)

def pick_splits(basenames: List[str], train_n: int, test_n: int, val_n: int, seed: int) -> Dict[str, List[str]]:
    """
    Randomly sample TRAIN/TEST/VAL splits by basename.
    """
    required = train_n + test_n + val_n
    if len(basenames) < required:
        raise ValueError(f"Not enough samples in instance folder: have {len(basenames)}, need {required}.")

    rnd = random.Random(seed)
    chosen = rnd.sample(basenames, required)

    train = chosen[:train_n]
    test  = chosen[train_n:train_n + test_n]
    val   = chosen[train_n + test_n:]

    return {"Training": train, "Testing": test, "Validation": val}

def copy_if_exists(src_file: Path, dst_file: Path, overwrite: bool = False) -> bool:
    """
    Copy src_file to dst_file if src exists.
    Returns True if copied (or skipped due to exists), False if source missing.
    """
    if not src_file.exists():
        return False
    if dst_file.exists() and not overwrite:
        return True  # treat as success (already present)
    dst_file.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src_file, dst_file)
    return True

def copy_one_basename(base_dir: Path, split_dirs: Dict[str, Dict[str, Path]], split: str, basename: str) -> Dict[str, bool]:
    """
    Copy all modalities for a single basename (without extension) into the split's destination.
    Attempts .tif first, then .tiff if needed.
    Returns a dict modality->bool indicating success for each.
    """
    results = {}
    for modality, rel in RELATIVE_FOLDERS.items():
        src_dir = base_dir / rel
        # prefer .tif, fallback .tiff
        src_tif  = src_dir / f"{basename}.tif"
        src_tiff = src_dir / f"{basename}.tiff"
        if src_tif.exists():
            src_path = src_tif
            ext = ".tif"
        elif src_tiff.exists():
            src_path = src_tiff
            ext = ".tiff"
        else:
            results[modality] = False
            continue

        dst_dir = split_dirs[split][modality]
        dst_path = dst_dir / f"{basename}{ext}"
        ok = copy_if_exists(src_path, dst_path, overwrite=OVERWRITE)
        results[modality] = ok
    return results

def print_folder_structure(folder: Path, indent: int = 0) -> None:
    """
    Print a clean, folders-only tree (matches your preferred style).
    """
    print("    " * indent + f"[{folder.name}]")
    try:
        for item in sorted(folder.iterdir()):
            if item.is_dir():
                print_folder_structure(item, indent + 1)
    except PermissionError:
        print("    " * (indent + 1) + "[Permission Denied]")

def summarize_split(base_name: str, split: str, counts: Dict[str, int]) -> None:
    """
    Print a compact summary for a split.
    """
    print(f"\n=== {base_name} — {split} Summary ===")
    total = sum(counts.values())
    print(f"Total copied entries (by modality success count): {total}")
    for modality in ["instance", "semantic_2class", "semantic_3_class", "window_a", "window_b"]:
        print(f"  {modality}: {counts.get(modality, 0)}")

# ----------------------------
# MAIN PROCESS
# ----------------------------
for base in BASE_FOLDERS:
    base_dir = Path(base)
    if not base_dir.exists():
        print(f"[WARN] Base folder does not exist, skipping: {base_dir}")
        continue

    # Prepare destination split directories (and subfolders)
    split_dirs = ensure_split_dirs(base_dir)

    # Get all instance basenames and sample splits
    basenames = list_instance_basenames(base_dir)
    splits = pick_splits(basenames, TRAIN_COUNT, TEST_COUNT, VAL_COUNT, RANDOM_SEED)

    # Copy files
    base_name = base_dir.name
    print(f"\nProcessing base: {base_dir}")
    for split_name, names in splits.items():
        # Track per-modality copy success counts
        success_counts = {k: 0 for k in RELATIVE_FOLDERS.keys()}

        for name in tqdm(names, desc=f"Copying {split_name} for {base_name}", unit="file"):
            results = copy_one_basename(base_dir, split_dirs, split_name, name)
            # Increment successes modality-wise
            for modality, ok in results.items():
                if ok:
                    success_counts[modality] += 1

        summarize_split(base_name, split_name, success_counts)

    # Show the created folder structures at the end
    print("\n=== Folder Structure ===")
    # Print only the three split roots to keep output tidy
    for split_root in [split_dirs["Training"]["instance"].parents[2],  # up to the split root
                       split_dirs["Testing"]["instance"].parents[2],
                       split_dirs["Validation"]["instance"].parents[2]]:
        print_folder_structure(split_root)



Processing base: C:\Users\grupp\Desktop\Mato Grosso AL MVP for tech note\MG AL true sets\MatoGrosso_Brazil_Confidences


Copying Training for MatoGrosso_Brazil_Confidences: 100%|██████████| 60/60 [00:01<00:00, 40.71file/s]



=== MatoGrosso_Brazil_Confidences — Training Summary ===
Total copied entries (by modality success count): 240
  instance: 60
  semantic_2class: 60
  semantic_3_class: 0
  window_a: 60
  window_b: 60


Copying Testing for MatoGrosso_Brazil_Confidences: 100%|██████████| 20/20 [00:00<00:00, 37.53file/s]



=== MatoGrosso_Brazil_Confidences — Testing Summary ===
Total copied entries (by modality success count): 80
  instance: 20
  semantic_2class: 20
  semantic_3_class: 0
  window_a: 20
  window_b: 20


...


=== MatoGrosso_Brazil_Random — Training Summary ===
Total copied entries (by modality success count): 240
  instance: 60
  semantic_2class: 60
  semantic_3_class: 0
  window_a: 60
  window_b: 60


Copying Testing for MatoGrosso_Brazil_Random: 100%|██████████| 20/20 [00:00<00:00, 33.45file/s]



=== MatoGrosso_Brazil_Random — Testing Summary ===
Total copied entries (by modality success count): 80
  instance: 20
  semantic_2class: 20
  semantic_3_class: 0
  window_a: 20
  window_b: 20


Copying Validation for MatoGrosso_Brazil_Random: 100%|██████████| 20/20 [00:00<00:00, 42.16file/s]



=== MatoGrosso_Brazil_Random — Validation Summary ===
Total copied entries (by modality success count): 80
  instance: 20
  semantic_2class: 20
  semantic_3_class: 0
  window_a: 20
  window_b: 20

=== Folder Structure ===
[MG AL true sets]
    [Chiquitano_Bolivia_Random]
        [label_masks]
            [instance]
            [semantic_2class]
            [semantic_3class]
        [s2_images]
            [window_a]
            [window_b]
    [MatoGrosso_Brazil_Confidences]
        [label_masks]
            [instance]
            [semantic_2class]
            [semantic_3class]
        [s2_images]
            [window_a]
            [window_b]
    [MatoGrosso_Brazil_Confidences_Testing]
        [label_masks]
            [instance]
            [semantic_2class]
            [semantic_3_class]
        [s2_images]
            [window_a]
            [window_b]
    [MatoGrosso_Brazil_Confidences_Training]
        [label_masks]
            [instance]
            [semantic_2class]
            [